# 03 - Your first calibration

Notebook 02 left the model a few percent above the measurements. One parameter
is responsible, and this notebook turns it until the curves meet.

The steps are always the same:

1. pick the parameters to fit,
2. pick the channels to fit them against,
3. give the optimiser a search interval,
4. look at the result *and* at the fit, not only at the number.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from demo_plant import DEFAULT_K_HYD_CH, TRUE_PARAMETERS, build_demo_plant, make_twin_measurements, simulate

from pyadm1ode_calibration import InitialCalibrator

measurements = make_twin_measurements(days=5, noise=0.02, seed=0)
observed = measurements.data["Q_gas"].to_numpy()

print("we are looking for:", TRUE_PARAMETERS)

## What the objective looks like

Before letting an optimiser loose, it is worth seeing the landscape it will
search. We simulate a handful of values and compute the normalised error for
each. Cheap here and it tells us immediately whether the parameter has any
influence at all.

In [ ]:
def nrmse(parameter_value):
    simulated = np.asarray(simulate(plant, measurements, {"k_hyd_ch": parameter_value})["Q_gas"], dtype=float)
    return float(np.sqrt(np.mean((simulated - observed) ** 2)) / observed.mean())

plant = build_demo_plant(days=5)
grid = [1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
errors = [nrmse(k) for k in grid]

for k, err in zip(grid, errors):
    print(f"  k_hyd_ch = {k:4.1f}   NRMSE = {err:6.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(grid, [e * 100 for e in errors], "o-")
ax.axvline(TRUE_PARAMETERS["k_hyd_ch"], color="tab:green", ls="--", label="true value")
ax.axvline(DEFAULT_K_HYD_CH, color="tab:red", ls=":", label="model default")
ax.set_xlabel("k_hyd_ch")
ax.set_ylabel("NRMSE [%]")
ax.set_title("The objective the optimiser has to minimise")
ax.legend()
fig.tight_layout()

## Running the calibration

`InitialCalibrator` wraps the whole loop. The arguments worth knowing:

| Argument | Meaning |
| --- | --- |
| `parameters` | which parameters to fit |
| `bounds` | search interval per parameter |
| `objectives` | which measured channels the error is computed on |
| `method` | `nelder_mead`, `powell`, `differential_evolution`, `slsqp`, `lbfgsb`, `particle_swarm` |
| `max_iterations` | budget; every iteration costs one simulation |
| `validation_split` | fraction held back from fitting |

In [ ]:
calibrator = InitialCalibrator(build_demo_plant(days=5), verbose=False)

result = calibrator.calibrate(
    measurements,
    parameters=["k_hyd_ch"],
    bounds={"k_hyd_ch": (1.0, 15.0)},
    objectives=["Q_gas"],
    method="nelder_mead",
    max_iterations=20,
    sensitivity_analysis=False,
)

print(f"success         : {result.success}")
print(f"k_hyd_ch found  : {result.parameters['k_hyd_ch']:.3f}")
print(f"true value      : {TRUE_PARAMETERS['k_hyd_ch']:.3f}")
print(f"objective       : {result.objective_value:.4f}  (normalised RMSE)")
print(f"simulations run : {len(result.history)}")

## Did it actually fit?

A number alone does not tell you whether the fit is good, plot it.

In [ ]:
before = np.asarray(simulate(plant, measurements, {"k_hyd_ch": DEFAULT_K_HYD_CH})["Q_gas"], dtype=float)
after = np.asarray(simulate(plant, measurements, result.parameters)["Q_gas"], dtype=float)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(measurements.data.index, observed, label="measured", alpha=0.6)
ax.plot(measurements.data.index, before, label=f"before (k_hyd_ch={DEFAULT_K_HYD_CH})", ls=":")
ax.plot(measurements.data.index, after, label=f"after  (k_hyd_ch={result.parameters['k_hyd_ch']:.2f})")
ax.set_ylabel("Q_gas [m3/d]")
ax.set_title("Before and after calibration")
ax.legend()
fig.tight_layout()

print(f"NRMSE before: {np.sqrt(np.mean((before - observed) ** 2)) / observed.mean():.2%}")
print(f"NRMSE after : {np.sqrt(np.mean((after - observed) ** 2)) / observed.mean():.2%}")

## The optimiser's path

`result.history` records every simulation the optimiser ran. Plotting it shows
whether it converged or ran out of budget.

In [ ]:
values = [step["parameters"]["k_hyd_ch"] for step in result.history]
objectives = [step["objective"] for step in result.history]

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(values, "o-")
axes[0].axhline(TRUE_PARAMETERS["k_hyd_ch"], color="tab:green", ls="--")
axes[0].set_xlabel("simulation")
axes[0].set_ylabel("k_hyd_ch")

finite = [min(o, 1.0) for o in objectives]   # failed trials are reported as 1e10
axes[1].plot(finite, "o-", color="tab:orange")
axes[1].set_xlabel("simulation")
axes[1].set_ylabel("objective (capped at 1.0)")

fig.suptitle("What the optimiser tried")
fig.tight_layout()